# 03 — Métricas de Sentimento

**Objetivo:** medir o impacto da fricção na experiência do cliente via análise de sentimento.  
**Ferramenta:** VADER (NLTK) — adequado para texto conversacional informal.

**Perguntas:**
1. Como o sentimento evolui ao longo das conversas? Há um padrão de trajetória?
2. Conversas com fricção (try-again) terminam com sentimento mais negativo?
3. Há um "dip" detectável no momento em que a primeira solução falha?
4. Qual o valor de baseline para o guardrail do A/B test?

**Limitação importante:** VADER foi treinado em texto social/review, não em chat de suporte.  
Expressões de resolução educada ("that worked, I must have entered it wrong") podem ter score
negativo por palavras isoladas ("wrong"). Usar os scores de forma **direcional**, não literal.

In [ ]:
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats

sys.path.insert(0, str(Path('..') / 'src'))
from sentiment import score_text, score_conversation

pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAW_DIR     = Path('../data/raw')
PROC_DIR    = Path('../data/processed')
FIGURES_DIR = Path('../reports/figures')

# Carregar dados processados com sentimento
df = pd.read_parquet(PROC_DIR / 'conversations_sentiment.parquet')
print(f"Conversas carregadas: {len(df):,}")
df[['sent_opening','sent_closing','sent_delta','sent_mean']].describe().round(3)

## 1. Sentimento Global: Opening vs Closing

In [ ]:
print("=== Sentimento médio global ===")
print(f"  Opening: {df['sent_opening'].mean():+.3f}")
print(f"  Closing: {df['sent_closing'].mean():+.3f}")
print(f"  Delta:   {df['sent_delta'].mean():+.3f}  (conversas melhoram em média)")

# t-test: closing > opening?
t, p = stats.ttest_rel(df['sent_closing'], df['sent_opening'])
print(f"\n  t-test (closing > opening): t={t:.2f}, p={p:.2e}")

## 2. Impacto da Fricção no Sentimento

In [ ]:
grp = df.groupby('has_friction')[['sent_opening','sent_closing','sent_delta','sent_mean']].agg(['mean','std'])
print("Com fricção (1) vs sem fricção (0):")
print(grp.round(3))

# Mann-Whitney: closing sentiment
no_friction = df[df['has_friction']==0]['sent_closing']
with_friction = df[df['has_friction']==1]['sent_closing']
u, p = stats.mannwhitneyu(no_friction, with_friction, alternative='greater')
print(f"\nMann-Whitney (closing, sem>com fricção): U={u:.0f}, p={p:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

grp_means = df.groupby('has_friction')[['sent_opening','sent_closing','sent_mean']].mean()
labels = ['Sem fricção', 'Com fricção']
pairs = [
    ('sent_opening', 'Sentimento Opening',  ['#aec7e8','#ffbb78']),
    ('sent_closing', 'Sentimento Closing',  ['#2ca02c','#d62728']),
    ('sent_mean',    'Sentimento Médio',    ['#aec7e8','#ffbb78']),
]

for ax, (col, title, colors) in zip(axes, pairs):
    vals = grp_means[col].values
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f'{val:+.3f}', ha='center', va='bottom', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, vals.max() * 1.3)
    ax.axhline(0, color='black', linewidth=0.5)

plt.suptitle('Impacto da Fricção no Sentimento do Cliente', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_sentiment_friction_impact.png', bbox_inches='tight')
plt.show()

## 3. Trajetória de Sentimento Intra-Conversa

Dividimos cada conversa em 5 blocos de igual proporção para comparar trajetórias de comprimentos diferentes.

In [ ]:
with open(RAW_DIR / 'abcd_v1.1.json', 'r', encoding='utf-8') as f:
    raw = json.load(f)
all_convos = [c for split, convos in raw.items() for c in convos]

def compute_trajectory(convos_subset):
    buckets = {i: [] for i in range(5)}
    for c in convos_subset:
        cust = [t[1] for t in c['original'] if t[0] == 'customer']
        n = len(cust)
        if n < 2:
            continue
        for rank, text in enumerate(cust):
            bucket = min(int(rank / n * 5), 4)
            buckets[bucket].append(score_text(text))
    return [np.mean(buckets[i]) if buckets[i] else 0.0 for i in range(5)]

sc_all   = [c for c in all_convos if c['scenario']['subflow'] == 'shopping_cart']
sc_nofr  = [c for c in sc_all if 'try-again' not in
            [t['targets'][2] for t in c['delexed'] if t['targets'][1]=='take_action']]
sc_fr    = [c for c in sc_all if 'try-again' in
            [t['targets'][2] for t in c['delexed'] if t['targets'][1]=='take_action']]

traj_nofr = compute_trajectory(sc_nofr)
traj_fr   = compute_trajectory(sc_fr)

print(f"SEM try-again (n={len(sc_nofr)}): {[f'{v:+.3f}' for v in traj_nofr]}")
print(f"COM try-again (n={len(sc_fr)}):  {[f'{v:+.3f}' for v in traj_fr]}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

x = ['Início', 'Quarto 1', 'Meio', 'Quarto 3', 'Fim']

ax.plot(x, traj_nofr, 'o-', color='#2ca02c', linewidth=2.5, markersize=8, label=f'Sem try-again (n={len(sc_nofr)})')
ax.plot(x, traj_fr,   's--', color='#d62728', linewidth=2.5, markersize=8, label=f'Com try-again (n={len(sc_fr)})')

# Marcar o dip
dip_idx = 2
ax.annotate('Dip: solução\nfalhou',
            xy=(x[dip_idx], traj_fr[dip_idx]),
            xytext=(x[dip_idx], traj_fr[dip_idx] - 0.05),
            arrowprops=dict(arrowstyle='->', color='#d62728'),
            ha='center', color='#d62728', fontsize=9)

ax.axhline(0, color='black', linewidth=0.5, linestyle=':')
ax.set_ylabel('Sentimento VADER (compound)')
ax.set_title('shopping_cart — Trajetória de Sentimento do Cliente', fontweight='bold')
ax.legend()
ax.set_ylim(-0.05, 0.35)

# Anotar diferença no closing
diff = traj_nofr[-1] - traj_fr[-1]
ax.annotate(f'Δ closing = {diff:+.3f}',
            xy=(x[-1], (traj_nofr[-1]+traj_fr[-1])/2),
            xytext=(3.5, (traj_nofr[-1]+traj_fr[-1])/2 + 0.03),
            fontsize=9, color='navy', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_sentiment_trajectory_shopping_cart.png', bbox_inches='tight')
plt.show()

## 4. Sentimento por Flow: Quem Começa e Termina Melhor?

In [ ]:
flow_sent = (
    df.groupby('flow')[['sent_opening','sent_closing','sent_delta']]
    .mean()
    .sort_values('sent_closing')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(flow_sent))
ax.bar([i-0.2 for i in x], flow_sent['sent_opening'], 0.35,
       label='Opening', color='#aec7e8', edgecolor='none')
ax.bar([i+0.2 for i in x], flow_sent['sent_closing'], 0.35,
       label='Closing', color='#2ca02c', edgecolor='none')

ax.set_xticks(list(x))
ax.set_xticklabels(flow_sent['flow'], rotation=25, ha='right')
ax.set_ylabel('Sentimento VADER (compound)')
ax.set_title('Opening vs Closing Sentiment por Flow', fontweight='bold')
ax.legend()
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_sentiment_by_flow.png', bbox_inches='tight')
plt.show()

print(flow_sent[['flow','sent_opening','sent_closing','sent_delta']].to_string(index=False))

## 5. Guardrail do A/B Test

O guardrail impede que a variante degrade a experiência. Definimos os valores de baseline aqui.

In [ ]:
sc_df = df[df['subflow'] == 'shopping_cart'].copy()

grp = sc_df.groupby('has_friction')[['sent_closing','sent_delta']].agg(['mean','std','count'])
print("=== shopping_cart: guardrail baseline ===")
print(grp.round(3))

baseline_closing = sc_df[sc_df['has_friction']==1]['sent_closing'].mean()
target_closing   = sc_df[sc_df['has_friction']==0]['sent_closing'].mean()
print(f"""
Controle (com try-again):     closing = {baseline_closing:+.3f}
Meta (sem try-again):         closing = {target_closing:+.3f}
Guardrail: Variante B não pode ter closing < {baseline_closing:+.3f}
""")

# IC 95% para o baseline do guardrail
n  = sc_df[sc_df['has_friction']==1]['sent_closing'].count()
se = sc_df[sc_df['has_friction']==1]['sent_closing'].std() / np.sqrt(n)
ci_low  = baseline_closing - 1.96 * se
ci_high = baseline_closing + 1.96 * se
print(f"IC 95% do baseline de closing: [{ci_low:+.3f}, {ci_high:+.3f}]")

## 6. Sumário do Notebook 03

### Achados

| Métrica | Sem fricção | Com fricção | Diferença |
|---|---|---|---|
| Sentimento opening (global) | +0.088 | +0.049 | -0.039 |
| Sentimento closing (global) | +0.255 | +0.220 | **-0.035** |
| Sentimento médio (global) | +0.155 | +0.126 | -0.029 |
| Mann-Whitney closing (sem>com) | | | **p < 0.0001** |

**shopping_cart especificamente:**
- Conversas sem try-again fecham em **+0.265**
- Conversas com try-again fecham em **+0.225** (Δ = -0.040)
- O "dip" de sentimento no bucket 2 (+0.096 vs +0.134) é o momento em que a primeira solução falha

### Guardrail para o A/B Test

> **Regra:** o sentimento de closing da Variante B não pode ser estatisticamente inferior ao baseline de **+0.225** (IC 95%: [+0.196, +0.255])
>
> Se a variante melhorar a resolução de 1ª tentativa mas degradar o sentimento de fechamento → **Kill**.

**Próximo passo:** notebook 04 — detecção automática de padrões estruturais (loops, escalações) para completar o diagnóstico antes do ICE scoring.